# DistilBERT (starter) — Jigsaw toxic comments

Same layout as `03_bert.ipynb` but **`preprocess_for_distilbert`** and **DistilBERT** weights (fewer parameters, faster).

**Quick iteration:** In the next cell, set `QUICK_ITERATION = True` for a small subset (same idea as `03_bert.ipynb`); `False` uses the full train/validation split.

**Requires:** `pip install torch transformers`

In [1]:
from pathlib import Path
import sys

# Repo root (contains preprocessing/)
ROOT = Path.cwd().resolve()
for c in (ROOT, ROOT.parent, ROOT.parent.parent, ROOT.parent.parent.parent):
    if (c / "preprocessing" / "text_preprocessing.py").exists():
        ROOT = c
        break
sys.path.insert(0, str(ROOT))
# notebooks/ for metrics_helpers
sys.path.insert(0, str(ROOT / "notebooks"))

In [2]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from preprocessing.text_preprocessing import LABEL_COLUMNS, preprocess_for_distilbert
from metrics_helpers import multilabel_evaluation_report, torch_parameter_count


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def binary_f1(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    denom = 2 * tp + fp + fn
    return float((2 * tp) / denom) if denom > 0 else 0.0


def tune_per_label_thresholds(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    labels: list[str] | tuple[str, ...],
    threshold_grid: np.ndarray,
) -> tuple[dict[str, float], pd.DataFrame]:
    best_thresholds: dict[str, float] = {}
    threshold_rows: list[dict[str, float | str]] = []

    for j, label in enumerate(labels):
        y_true_j = y_true[:, j]
        y_prob_j = y_prob[:, j]

        best_t = 0.5
        best_f1 = -1.0
        for t in threshold_grid:
            y_pred_j = (y_prob_j >= t).astype(np.int64)
            f1_j = binary_f1(y_true_j, y_pred_j)
            if f1_j > best_f1:
                best_f1 = f1_j
                best_t = float(t)

        best_thresholds[label] = best_t
        threshold_rows.append(
            {
                "label": label,
                "best_threshold": round(best_t, 3),
                "best_f1_on_val": best_f1,
            }
        )

    threshold_df = pd.DataFrame(threshold_rows)
    return best_thresholds, threshold_df


def apply_thresholds(y_prob: np.ndarray, labels: list[str] | tuple[str, ...], thresholds: dict[str, float]) -> np.ndarray:
    out = np.zeros_like(y_prob, dtype=np.int64)
    for j, label in enumerate(labels):
        out[:, j] = (y_prob[:, j] >= thresholds[label]).astype(np.int64)
    return out


def enc_dict(enc):
    keys = [k for k in ("input_ids", "attention_mask") if k in enc]
    return {k: enc[k] for k in keys}


def compute_pos_weight(y_train: np.ndarray, eps: float = 1e-6, max_weight: float = 50.0) -> torch.Tensor:
    y_train = np.asarray(y_train, dtype=np.float32)
    positives = y_train.sum(axis=0)
    total = float(y_train.shape[0])
    negatives = total - positives
    weights = negatives / np.maximum(positives, eps)
    weights = np.clip(weights, 1.0, max_weight)
    return torch.tensor(weights, dtype=torch.float32)


class EncDataset(Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.labels.shape[0]

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


def collate(batch):
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0] if k != "labels"}
    out["labels"] = torch.stack([b["labels"] for b in batch], dim=0)
    return out


DEVICE = pick_device()
print("Device:", DEVICE)
torch.manual_seed(42)

QUICK_ITERATION = False

if QUICK_ITERATION:
    MAX_LENGTH = 64
    BATCH_SIZE = 16
    max_train_cap = 30000
    MAX_VAL_SAMPLES = 3000
else:
    MAX_LENGTH = 128
    BATCH_SIZE = 8
    max_train_cap = None
    MAX_VAL_SAMPLES = None

EPOCHS = 2
LR = 2e-5
SIZE_STEP = 10000
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)

probe_data = preprocess_for_distilbert(
    validation_fraction=0.1,
    random_state=42,
    max_length=MAX_LENGTH,
    return_tensors="pt",
    max_train_samples=max_train_cap,
    max_val_samples=MAX_VAL_SAMPLES,
)

max_train_available = len(probe_data.y_train)
if max_train_available < SIZE_STEP:
    train_sizes = [max_train_available]
else:
    train_sizes = list(range(SIZE_STEP, max_train_available + 1, SIZE_STEP))
    if train_sizes[-1] != max_train_available:
        train_sizes.append(max_train_available)

print("CONFIG")
print("  mode:", "quick" if QUICK_ITERATION else "full-data")
print("  device:", DEVICE)
print("  max_train_available:", max_train_available)
print("  train_sizes:", train_sizes[:5], "...", train_sizes[-1], f"(n={len(train_sizes)})")
print("  max_length:", MAX_LENGTH, "| batch_size:", BATCH_SIZE)
print("  epochs per size:", EPOCHS)
print("  lr:", LR)

summary_records: list[dict[str, float | int]] = []
per_label_records: list[pd.DataFrame] = []
threshold_records: list[pd.DataFrame] = []

start_total = time.perf_counter()
for i, train_size in enumerate(train_sizes, start=1):
    print(f"\n=== Benchmark {i}/{len(train_sizes)} | train_size={train_size} ===")
    run_data = preprocess_for_distilbert(
        validation_fraction=0.1,
        random_state=42,
        max_length=MAX_LENGTH,
        return_tensors="pt",
        max_train_samples=train_size,
        max_val_samples=MAX_VAL_SAMPLES,
    )

    train_enc = enc_dict(run_data.train_encodings)
    val_enc = enc_dict(run_data.val_encodings)
    y_train = np.asarray(run_data.y_train, dtype=np.float32)
    y_val = np.asarray(run_data.y_val, dtype=np.int64)

    train_loader = DataLoader(EncDataset(train_enc, y_train), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(EncDataset(val_enc, y_val), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate)

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        num_labels=len(LABEL_COLUMNS),
        problem_type="multi_label_classification",
    ).to(DEVICE)
    num_params = torch_parameter_count(model)

    pos_weight = compute_pos_weight(y_train).to(DEVICE)
    pos_weight_dict = {label: float(pos_weight[idx].item()) for idx, label in enumerate(LABEL_COLUMNS)}

    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    run_start = time.perf_counter()
    train_loss_last = np.nan
    val_loss_last = np.nan
    train_time_s = 0.0
    inference_time_s = 0.0
    summary_05: dict[str, float] | None = None
    summary_tuned: dict[str, float] | None = None
    per_label_compare_df = pd.DataFrame()
    threshold_df = pd.DataFrame()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss_sum = 0.0
        epoch_batches = 0

        train_epoch_start = time.perf_counter()
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")

            opt.zero_grad()
            out = model(**batch)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            opt.step()

            epoch_loss_sum += float(loss.item())
            epoch_batches += 1

        train_time_s += time.perf_counter() - train_epoch_start
        train_loss_last = epoch_loss_sum / max(epoch_batches, 1)

        model.eval()
        all_probs: list[np.ndarray] = []
        val_loss_sum = 0.0
        val_batches = 0

        inf_start = time.perf_counter()
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = batch.pop("labels")
                logits = model(**batch).logits
                probs = torch.sigmoid(logits)
                all_probs.append(probs.cpu().numpy())

                val_loss_sum += float(loss_fn(logits, labels).item())
                val_batches += 1

        inference_time_s += time.perf_counter() - inf_start
        val_loss_last = val_loss_sum / max(val_batches, 1)

        prob_val = np.concatenate(all_probs, axis=0)
        pred_val_05 = (prob_val >= 0.5).astype(np.int64)

        per_label_05_df, summary_05 = multilabel_evaluation_report(y_val, pred_val_05, prob_val, LABEL_COLUMNS)

        best_thresholds, threshold_df = tune_per_label_thresholds(y_val, prob_val, LABEL_COLUMNS, THRESHOLD_GRID)
        pred_val_tuned = apply_thresholds(prob_val, LABEL_COLUMNS, best_thresholds)
        per_label_tuned_df, summary_tuned = multilabel_evaluation_report(y_val, pred_val_tuned, prob_val, LABEL_COLUMNS)

        per_label_compare_df = per_label_05_df.rename(
            columns={
                "precision": "precision_baseline",
                "recall": "recall_baseline",
                "f1": "f1_baseline",
                "roc_auc": "roc_auc_baseline",
            }
        ).merge(
            per_label_tuned_df.rename(
                columns={
                    "precision": "precision_tuned",
                    "recall": "recall_tuned",
                    "f1": "f1_tuned",
                    "roc_auc": "roc_auc_tuned",
                }
            ),
            on="label",
            how="left",
        )

        print(
            f"  Epoch {epoch}/{EPOCHS} | train_loss={train_loss_last:.4f} | "
            f"val_loss={val_loss_last:.4f} | train_s={train_time_s:.1f} | infer_s={inference_time_s:.1f}"
        )

    assert summary_05 is not None and summary_tuned is not None

    per_label_compare_df["train_size"] = train_size
    per_label_compare_df["epoch"] = EPOCHS
    for label in LABEL_COLUMNS:
        per_label_compare_df[f"pos_weight_{label}"] = pos_weight_dict[label]

    threshold_df["train_size"] = train_size
    threshold_df["epoch"] = EPOCHS

    run_elapsed = time.perf_counter() - run_start
    summary_records.append(
        {
            "train_size": train_size,
            "epochs": EPOCHS,
            "max_length": MAX_LENGTH,
            "batch_size": BATCH_SIZE,
            "lr": LR,
            "num_parameters": num_params,
            "train_loss": train_loss_last,
            "val_loss": val_loss_last,
            "train_time_s": train_time_s,
            "inference_time_s": inference_time_s,
            "run_time_total_s": run_elapsed,
            "f1_micro_baseline": summary_05["f1_micro"],
            "f1_macro_baseline": summary_05["f1_macro"],
            "f1_micro_tuned": summary_tuned["f1_micro"],
            "f1_macro_tuned": summary_tuned["f1_macro"],
            "pos_weight_min": float(pos_weight.min().item()),
            "pos_weight_max": float(pos_weight.max().item()),
            "pos_weight_mean": float(pos_weight.mean().item()),
        }
    )

    per_label_records.append(per_label_compare_df)
    threshold_records.append(threshold_df)

benchmark_summary_df = pd.DataFrame(summary_records)
benchmark_per_label_df = pd.concat(per_label_records, ignore_index=True)
benchmark_thresholds_df = pd.concat(threshold_records, ignore_index=True)

artifact_dir = ROOT / "notebooks" / "artifacts"
artifact_dir.mkdir(parents=True, exist_ok=True)
summary_csv_path = artifact_dir / "distilbert_size_benchmark_summary.csv"
per_label_csv_path = artifact_dir / "distilbert_size_benchmark_per_label.csv"
threshold_csv_path = artifact_dir / "distilbert_size_benchmark_thresholds.csv"

benchmark_summary_df.to_csv(summary_csv_path, index=False)
benchmark_per_label_df.to_csv(per_label_csv_path, index=False)
benchmark_thresholds_df.to_csv(threshold_csv_path, index=False)

elapsed_total = time.perf_counter() - start_total
print("\nTraining-size benchmark complete")
print("  total_elapsed_s:", round(elapsed_total, 2))
print("  summary_csv:", summary_csv_path)
print("  per_label_csv:", per_label_csv_path)
print("  thresholds_csv:", threshold_csv_path)
print("\nBenchmark summary:")
display(benchmark_summary_df)

Device: mps
CONFIG
  mode: full-data
  device: mps
  max_train_available: 143613
  train_sizes: [10000, 20000, 30000, 40000, 50000] ... 143613 (n=15)
  max_length: 128 | batch_size: 8
  epochs per size: 2
  lr: 2e-05

=== Benchmark 1/15 | train_size=10000 ===


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/2 | train_loss=0.3954 | val_loss=0.2997 | train_s=222.4 | infer_s=83.4
  Epoch 2/2 | train_loss=0.2079 | val_loss=0.3348 | train_s=449.4 | infer_s=166.4

=== Benchmark 2/15 | train_size=20000 ===


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/2 | train_loss=0.3350 | val_loss=0.2544 | train_s=453.6 | infer_s=83.2
  Epoch 2/2 | train_loss=0.1952 | val_loss=0.2993 | train_s=904.7 | infer_s=166.4

=== Benchmark 3/15 | train_size=30000 ===


...

  Epoch 1/2 | train_loss=0.2773 | val_loss=0.2261 | train_s=1368.4 | infer_s=82.8
  Epoch 2/2 | train_loss=0.1767 | val_loss=0.2557 | train_s=2730.6 | infer_s=165.4

=== Benchmark 7/15 | train_size=70000 ===


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Epoch 1/2 | train_loss=0.2657 | val_loss=0.2168 | train_s=1596.5 | infer_s=82.6
  Epoch 2/2 | train_loss=0.1752 | val_loss=0.2562 | train_s=3203.9 | infer_s=165.8

=== Benchmark 8/15 | train_size=80000 ===


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


KeyboardInterrupt: 

In [ ]:
benchmark_metrics_df = benchmark_summary_df.copy()
benchmark_metrics_df["f1_micro_delta_tuned_minus_baseline"] = (
    benchmark_metrics_df["f1_micro_tuned"] - benchmark_metrics_df["f1_micro_baseline"]
)
benchmark_metrics_df["f1_macro_delta_tuned_minus_baseline"] = (
    benchmark_metrics_df["f1_macro_tuned"] - benchmark_metrics_df["f1_macro_baseline"]
)

print("Training-size comparison table:")
display(
    benchmark_metrics_df[
        [
            "train_size",
            "epochs",
            "train_loss",
            "val_loss",
            "train_time_s",
            "inference_time_s",
            "num_parameters",
            "f1_micro_baseline",
            "f1_macro_baseline",
            "f1_micro_tuned",
            "f1_macro_tuned",
            "f1_micro_delta_tuned_minus_baseline",
            "f1_macro_delta_tuned_minus_baseline",
            "pos_weight_min",
            "pos_weight_max",
            "pos_weight_mean",
        ]
    ]
)

print("\nSanity checks:")
expected_sizes = benchmark_metrics_df["train_size"].tolist()
print("  benchmark train sizes:", expected_sizes[:5], "...", expected_sizes[-1], f"(n={len(expected_sizes)})")
print("  summary rows:", len(benchmark_summary_df))
print("  per-label rows:", len(benchmark_per_label_df), "(expected", len(expected_sizes) * len(LABEL_COLUMNS), ")")
print("  threshold rows:", len(benchmark_thresholds_df), "(expected", len(expected_sizes) * len(LABEL_COLUMNS), ")")

all_pos_weight_valid = np.isfinite(
    benchmark_metrics_df[["pos_weight_min", "pos_weight_max", "pos_weight_mean"]].to_numpy()
).all() and (benchmark_metrics_df["pos_weight_min"] >= 0).all()
print("  pos_weight finite/nonnegative:", bool(all_pos_weight_valid))

print("\nSample per-label metrics (largest train_size):")
max_size = int(benchmark_per_label_df["train_size"].max())
max_size_per_label = (
    benchmark_per_label_df.loc[benchmark_per_label_df["train_size"] == max_size]
    .sort_values("label")
    .reset_index(drop=True)
)
display(
    max_size_per_label[
        [
            "label",
            "precision_baseline",
            "recall_baseline",
            "f1_baseline",
            "precision_tuned",
            "recall_tuned",
            "f1_tuned",
            "roc_auc_tuned",
        ]
    ]
)

print("\nSample thresholds (largest train_size):")
max_size_thresholds = (
    benchmark_thresholds_df.loc[benchmark_thresholds_df["train_size"] == max_size]
    .sort_values("label")
    .reset_index(drop=True)
)
display(max_size_thresholds)

In [ ]:
import matplotlib.pyplot as plt

# 1) Losses vs train size
plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["train_loss"], marker="o", label="train_loss")
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["val_loss"], marker="o", label="val_loss")
plt.title("DistilBERT: Loss vs Train Size")
plt.xlabel("Train size")
plt.ylabel("BCEWithLogits loss")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 2) Aggregate F1 vs train size
plt.figure(figsize=(9, 5))
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["f1_micro_baseline"], marker="o", label="micro F1 (0.5)")
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["f1_macro_baseline"], marker="o", label="macro F1 (0.5)")
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["f1_micro_tuned"], marker="o", linestyle="--", label="micro F1 (tuned)")
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["f1_macro_tuned"], marker="o", linestyle="--", label="macro F1 (tuned)")
plt.title("DistilBERT: Aggregate F1 vs Train Size")
plt.xlabel("Train size")
plt.ylabel("F1")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 3) Time vs train size
plt.figure(figsize=(8, 4.5))
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["train_time_s"], marker="o", label="train_time_s")
plt.plot(benchmark_metrics_df["train_size"], benchmark_metrics_df["inference_time_s"], marker="o", label="inference_time_s")
plt.title("DistilBERT: Runtime vs Train Size")
plt.xlabel("Train size")
plt.ylabel("Seconds")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 4) Threshold-tuning gain vs train size
plt.figure(figsize=(8, 4.5))
plt.plot(
    benchmark_metrics_df["train_size"],
    benchmark_metrics_df["f1_micro_delta_tuned_minus_baseline"],
    marker="o",
    label="delta micro F1 (tuned-baseline)",
)
plt.plot(
    benchmark_metrics_df["train_size"],
    benchmark_metrics_df["f1_macro_delta_tuned_minus_baseline"],
    marker="o",
    label="delta macro F1 (tuned-baseline)",
)
plt.axhline(0.0, color="black", linewidth=1, linestyle="--")
plt.title("DistilBERT: Threshold Tuning Gain vs Train Size")
plt.xlabel("Train size")
plt.ylabel("F1 delta")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

# 5) Per-label tuned F1 trend vs train size
plt.figure(figsize=(10, 5.5))
for label in LABEL_COLUMNS:
    label_df = benchmark_per_label_df.loc[benchmark_per_label_df["label"] == label].sort_values("train_size")
    plt.plot(label_df["train_size"], label_df["f1_tuned"], marker="o", label=f"{label} (tuned)")
plt.title("DistilBERT: Per-Label Tuned F1 vs Train Size")
plt.xlabel("Train size")
plt.ylabel("F1")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(loc="best")
plt.show()

# 6) Per-label best threshold trend vs train size
plt.figure(figsize=(10, 5.5))
for label in LABEL_COLUMNS:
    label_df = benchmark_thresholds_df.loc[benchmark_thresholds_df["label"] == label].sort_values("train_size")
    plt.plot(label_df["train_size"], label_df["best_threshold"], marker="o", label=label)
plt.title("DistilBERT: Tuned Threshold vs Train Size")
plt.xlabel("Train size")
plt.ylabel("Best threshold")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.legend(loc="best")
plt.show()

best_size_idx = benchmark_metrics_df["f1_macro_tuned"].idxmax()
best_size_row = benchmark_metrics_df.loc[best_size_idx]

print("--- DistilBERT train-size benchmark summary ---")
print(f"  mode: {'quick' if QUICK_ITERATION else 'full-data'}")
print(f"  sizes evaluated: {len(benchmark_metrics_df)}")
print(f"  epochs per size: {EPOCHS}")
print(f"  device: {DEVICE}")
print(f"  best_train_size_by_macro_tuned: {int(best_size_row['train_size'])}")
print(f"  best_macro_F1_tuned: {best_size_row['f1_macro_tuned']:.4f}")
print(
    "  threshold tuning nonnegative macro delta for all sizes:",
    bool((benchmark_metrics_df["f1_macro_delta_tuned_minus_baseline"] >= 0).all()),
)